# Clustering PCA + UMAP + LEIDEN

Notebook per caricare embedding e metadata CSV, applicare PCA al 95%, ottimizzare UMAP con `src/metrics/umap_metrics.py` e clusterizzare con LEIDEN`src/clustering/leiden.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.gmm import evaluate_gmm_clustered_data, optimize_gmm_bic_parameters
from src.metrics.umap_metrics import evaluate_umap_embedding, optimize_umap_parameters
from src.visualization.sample import show_clustering_samples

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## Caricamento embedding

Scegli il file `.npy` modificando `EMBEDDING_PATH`. Il CSV deve avere lo stesso nome e suffisso `.csv`.

In [ ]:
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "glomeruli" / "embeddings"
embedding_files = sorted(EMBEDDINGS_DIR.glob("*.npy"))

if not embedding_files:
    raise FileNotFoundError(f"Nessun file .npy trovato in {EMBEDDINGS_DIR}")

display(pd.DataFrame({"file": [path.name for path in embedding_files]}))

In [ ]:
EMBEDDING_PATH = EMBEDDINGS_DIR / "nasnet_masked_embeddings.npy"

if not EMBEDDING_PATH.exists():
    EMBEDDING_PATH = embedding_files[0]

CSV_PATH = EMBEDDING_PATH.with_suffix(".csv")

embeddings = np.load(EMBEDDING_PATH)
metadata = pd.read_csv(CSV_PATH)

if "index" in metadata.columns:
    metadata = metadata.sort_values("index").reset_index(drop=True)
else:
    metadata = metadata.reset_index(drop=True)

if len(metadata) != embeddings.shape[0]:
    raise ValueError(
        f"CSV e embedding non allineati: {len(metadata)} righe metadata, "
        f"{embeddings.shape[0]} embedding."
    )

if "image_path" in metadata.columns:
    image_paths = [
        path if path.is_absolute() else PROJECT_ROOT / path
        for path in metadata["image_path"].map(Path)
    ]
else:
    image_paths = []

print("Embedding:", EMBEDDING_PATH.name)
print("CSV:", CSV_PATH.name)
print("Shape embeddings:", embeddings.shape)
display(metadata.head())

## Preprocessing e PCA 95%

Gli embedding vengono standardizzati e poi ridotti con PCA mantenendo il 95% della varianza.

In [ ]:
PCA_VARIANCE = 0.95

scaler = StandardScaler()
X_scaled = scaler.fit_transform(embeddings)

pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print("Shape originale:", embeddings.shape)
print("Shape dopo PCA:", X_pca.shape)
print("Varianza spiegata:", pca.explained_variance_ratio_.sum())

## Ottimizzazione UMAP

La griglia sotto e' volutamente compatta. Aumenta i valori quando vuoi fare una ricerca piu' completa.

In [ ]:
umap_optimization = optimize_umap_parameters(
    X_pca,
    umap_n_neighbors_values=(10, 15, 20, 30, 40),
    umap_min_dist_values=(0.0, 0.05, 0.1, 0.3),
    umap_n_components_values=(2, 3, 5, 10, 15),
    umap_metric_values=("euclidean",),
    k_values=(5, 10, 15),
    random_state=RANDOM_STATE,
)

umap_results_df = umap_optimization["results"]
display(umap_results_df.head(10))

In [ ]:
best_umap_embedding = umap_optimization["best_embedding"]
best_umap_params = umap_optimization["best_params"]
best_umap_metrics = umap_optimization["best_metrics"]

if best_umap_embedding is None:
    raise RuntimeError("Nessuna configurazione UMAP valida trovata. Riduci k_values o amplia la griglia.")

print("Best UMAP params:")
display(best_umap_params)

print("Best UMAP metrics:")
display(pd.Series(best_umap_metrics))

umap_check_metrics = evaluate_umap_embedding(
    X_pca,
    best_umap_embedding,
    k_values=best_umap_params["evaluation"]["k_values"],
    source_metric=best_umap_params["umap"]["metric"],
    embedded_metric=best_umap_params["evaluation"]["embedded_metric"],
)
display(pd.Series(umap_check_metrics).rename("check_metrics"))

## Clustering LEIDEN con ottimizzazione

Il LEIDEN viene fitatto sul miglior embedding UMAP.

In [ ]:
from src.clustering.leiden import optimize_leiden_parameters_optuna

n_embeddings = best_umap_embedding.shape[0]

max_clusters = int(np.ceil(np.log2(n_embeddings)))
max_clusters = max(4, max_clusters)
max_clusters = min(12, max_clusters)
min_clusters = round(max_clusters / 2) + 1

leiden_optimization = optimize_leiden_parameters_optuna(
    X=best_umap_embedding,
    leiden_n_neighbors_values=(15, 20, 30, 40, 50),
    leiden_resolution_values=(0.4, 0.6, 0.8, 1.0, 1.2, 1.5),
    leiden_metric_values=("euclidean",),
    leiden_weighted_values=(True,),
    leiden_partition_type_values=("RBConfiguration",),
    min_clusters=min_clusters,
    max_clusters=max_clusters,
    min_cluster_size=10,
    random_state=RANDOM_STATE,
    n_trials=100,
    n_iterations=10,
    primary_metric="silhouette",
    multiobjective_weights={
        "silhouette": 0.40,
        "davies_bouldin": 0.30,
        "calinski_harabasz": 0.20,
        "quality": 0.10,
    },
)
leiden_labels = leiden_optimization["best_labels"]

leiden_optimization["best_params"]

In [ ]:
from src.clustering.leiden import evaluate_leiden_clustered_data

best_labels = leiden_optimization["best_labels"]
best_leiden_clustering = leiden_optimization["best_clustering"]
best_leiden_params = leiden_optimization["best_params"]

if best_labels is None:
    raise RuntimeError(
        "Nessuna configurazione Leiden valida trovata. "
        "Amplia la griglia o riduci min_cluster_size."
    )

best_leiden_metrics = evaluate_leiden_clustered_data(
    X=best_umap_embedding,
    labels=best_labels,
    quality=best_leiden_clustering["quality"],
    silhouette_metric="euclidean",
)

print("Best Leiden params:")
display(best_leiden_params)

print("Best Leiden metrics:")
display(pd.Series(best_leiden_metrics))

cluster_counts = (
    pd.Series(best_labels, name="cluster")
    .value_counts()
    .sort_index()
)

display(cluster_counts.rename("n_samples").to_frame())

display(
    leiden_optimization["selected_results"][
        [
            "rank_selection",
            "selection_score",
            "n_neighbors",
            "resolution",
            "metric",
            "weighted",
            "partition_type",
            "n_clusters",
            "min_cluster_size",
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz",
            "quality",
        ]
    ].head(20)
)

## Visualizzazione UMAP

Se il miglior UMAP ha almeno 2 componenti, visualizza le prime due componenti colorate per cluster GMM.

In [ ]:
if best_umap_embedding.shape[1] < 2:
    raise ValueError("Servono almeno 2 componenti UMAP per lo scatter plot.")

figure, axis = plt.subplots(figsize=(7, 6))
scatter = axis.scatter(
    best_umap_embedding[:, 0],
    best_umap_embedding[:, 1],
    c=best_labels,
    cmap="tab20",
    s=20,
    alpha=0.85,
    edgecolors="none",
)
axis.set_title(f"{EMBEDDING_PATH.stem} - GMM su UMAP ottimizzato")
axis.set_xlabel("UMAP 1")
axis.set_ylabel("UMAP 2")
axis.grid(True, alpha=0.2)
figure.colorbar(scatter, ax=axis, label="Cluster")
figure.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image


def plot_clickable_umap_with_images(
    embedding,
    labels,
    image_paths,
    clusters_to_show,
    title=None,
    point_size_other=12,
    point_size_selected=30,
    click_only_selected=True,
    max_click_distance_px=15,
):
    """
    Scatter plot UMAP cliccabile.

    Quando clicchi vicino a un punto, viene aperta l'immagine corrispondente.

    Questa versione usa button_press_event ed è più robusta di pick_event.
    """

    embedding = np.asarray(embedding)
    labels = np.asarray(labels)
    image_paths = [Path(p) for p in image_paths]

    if embedding.ndim != 2:
        raise ValueError("embedding deve essere un array 2D.")

    if embedding.shape[1] < 2:
        raise ValueError("Servono almeno 2 componenti UMAP.")

    if len(labels) != embedding.shape[0]:
        raise ValueError(
            f"labels ha lunghezza {len(labels)}, "
            f"ma embedding ha {embedding.shape[0]} righe."
        )

    if len(image_paths) != embedding.shape[0]:
        raise ValueError(
            f"image_paths ha lunghezza {len(image_paths)}, "
            f"ma embedding ha {embedding.shape[0]} righe."
        )

    clusters_to_show = list(clusters_to_show)

    selected_mask = np.isin(labels, clusters_to_show)
    other_mask = ~selected_mask

    selected_indices = np.where(selected_mask)[0]
    other_indices = np.where(other_mask)[0]

    figure, axis = plt.subplots(figsize=(7, 6))

    axis.scatter(
        embedding[other_mask, 0],
        embedding[other_mask, 1],
        c="lightgray",
        s=point_size_other,
        alpha=0.25,
        edgecolors="none",
        label="Altri cluster",
    )

    scatter_selected = axis.scatter(
        embedding[selected_mask, 0],
        embedding[selected_mask, 1],
        c=labels[selected_mask],
        cmap="tab20",
        s=point_size_selected,
        alpha=0.9,
        edgecolors="none",
        label=f"Cluster {clusters_to_show}",
    )

    if title is None:
        title = f"Cluster evidenziati: {clusters_to_show}"

    axis.set_title(title)
    axis.set_xlabel("UMAP 1")
    axis.set_ylabel("UMAP 2")
    axis.grid(True, alpha=0.2)
    axis.legend()

    if len(selected_indices) > 0:
        figure.colorbar(
            scatter_selected,
            ax=axis,
            label="Cluster selezionati",
        )

    if click_only_selected:
        clickable_indices = selected_indices
    else:
        clickable_indices = np.arange(embedding.shape[0])

    clickable_xy = embedding[clickable_indices, :2]

    def show_image(global_index):
        image_path = image_paths[global_index]

        if not image_path.exists():
            print(f"Immagine non trovata: {image_path}")
            return

        image = Image.open(image_path).convert("RGB")

        image_figure, image_axis = plt.subplots(figsize=(5, 5))
        image_axis.imshow(image)
        image_axis.axis("off")

        image_axis.set_title(
            f"idx={global_index} | cluster={labels[global_index]}\n"
            f"{image_path.name}"
        )

        image_figure.tight_layout()
        image_figure.canvas.draw_idle()
        image_figure.show()

    def on_click(event):
        if event.inaxes != axis:
            return

        if len(clickable_indices) == 0:
            return

        # Coordinate dei punti in pixel dello schermo
        point_pixels = axis.transData.transform(clickable_xy)

        click_pixel = np.array([event.x, event.y])

        distances = np.linalg.norm(
            point_pixels - click_pixel,
            axis=1,
        )

        nearest_local_index = int(np.argmin(distances))
        nearest_distance = distances[nearest_local_index]

        if nearest_distance > max_click_distance_px:
            print(
                "Click troppo lontano da un punto. "
                f"Distanza: {nearest_distance:.1f}px"
            )
            return

        global_index = int(clickable_indices[nearest_local_index])

        print(
            f"Clicked idx={global_index}, "
            f"cluster={labels[global_index]}, "
            f"path={image_paths[global_index].name}"
        )

        show_image(global_index)

    callback_id = figure.canvas.mpl_connect(
        "button_press_event",
        on_click,
    )

    # Evita che il callback venga eliminato dal garbage collector
    figure._click_callback = on_click

    figure.tight_layout()
    plt.show()

    return figure, axis, callback_id

In [ ]:
#%matplotlib widget

import matplotlib
import matplotlib.pyplot as plt

print(matplotlib.get_backend())

clusters_to_show = [0, 1]

figure, axis, callback_id = plot_clickable_umap_with_images(
    embedding=best_umap_embedding,
    labels=best_labels,
    image_paths=image_paths,
    clusters_to_show=clusters_to_show,
    title=(
        f"{EMBEDDING_PATH.stem} - GMM su UMAP ottimizzato\n"
        f"Cluster evidenziati: {clusters_to_show}"
    ),
    click_only_selected=True,
    max_click_distance_px=15,
)

In [ ]:
import base64
from io import BytesIO
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from PIL import Image


def image_to_data_uri(image_path, max_size=420):
    """
    Converte un'immagine in data URI base64 per inserirla nell'HTML.
    """

    image_path = Path(image_path)

    image = Image.open(image_path).convert("RGB")
    image.thumbnail((max_size, max_size))

    buffer = BytesIO()
    image.save(buffer, format="PNG")

    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return f"data:image/png;base64,{encoded}"


def create_clickable_umap_html(
    embedding,
    labels,
    image_paths,
    clusters_to_show,
    output_html_path="clickable_umap.html",
    title="Clickable UMAP",
    click_only_selected=True,
    max_image_size=420,
):
    """
    Crea un file HTML interattivo.

    Cliccando su un punto evidenziato, viene mostrata l'immagine associata.
    """

    embedding = np.asarray(embedding)
    labels = np.asarray(labels)
    image_paths = [Path(p) for p in image_paths]

    if embedding.shape[1] < 2:
        raise ValueError("Servono almeno 2 componenti UMAP.")

    if len(labels) != embedding.shape[0]:
        raise ValueError(
            f"labels ha lunghezza {len(labels)}, "
            f"ma embedding ha {embedding.shape[0]} righe."
        )

    if len(image_paths) != embedding.shape[0]:
        raise ValueError(
            f"image_paths ha lunghezza {len(image_paths)}, "
            f"ma embedding ha {embedding.shape[0]} righe."
        )

    clusters_to_show = list(clusters_to_show)

    selected_mask = np.isin(labels, clusters_to_show)
    other_mask = ~selected_mask

    selected_indices = np.where(selected_mask)[0]
    other_indices = np.where(other_mask)[0]

    if click_only_selected:
        clickable_mask = selected_mask
    else:
        clickable_mask = np.ones(len(labels), dtype=bool)

    # Preparo customdata per i punti selezionati
    selected_customdata = []

    for index in selected_indices:
        image_path = image_paths[index]

        if image_path.exists():
            image_data = image_to_data_uri(
                image_path,
                max_size=max_image_size,
            )
        else:
            image_data = ""

        selected_customdata.append(
            [
                image_data,
                str(image_path.name),
                int(index),
                int(labels[index]),
                str(image_path),
            ]
        )

    fig = go.Figure()

    # Altri cluster
    fig.add_trace(
        go.Scatter(
            x=embedding[other_mask, 0],
            y=embedding[other_mask, 1],
            mode="markers",
            name="Altri cluster",
            marker=dict(
                size=6,
                color="lightgray",
                opacity=0.35,
            ),
            hoverinfo="skip",
        )
    )

    # Cluster evidenziato
    fig.add_trace(
        go.Scatter(
            x=embedding[selected_mask, 0],
            y=embedding[selected_mask, 1],
            mode="markers",
            name=f"Cluster {clusters_to_show}",
            marker=dict(
                size=9,
                color=labels[selected_mask],
                colorscale="Turbo",
                showscale=True,
                opacity=0.95,
            ),
            customdata=selected_customdata,
            hovertemplate=(
                "idx=%{customdata[2]}<br>"
                "cluster=%{customdata[3]}<br>"
                "%{customdata[1]}"
                "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="UMAP 1",
        yaxis_title="UMAP 2",
        width=850,
        height=700,
        template="plotly_white",
    )

    div_id = "umap_plot"

    plot_html = pio.to_html(
        fig,
        include_plotlyjs=True,
        full_html=False,
        div_id=div_id,
    )

    html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{title}</title>
</head>
<body>

<h2>{title}</h2>

<div style="display: flex; gap: 25px; align-items: flex-start;">

    <div>
        {plot_html}
    </div>

    <div style="width: 450px;">
        <h3 id="preview_title">Clicca su un punto evidenziato</h3>

        <img
            id="preview_img"
            src=""
            style="
                max-width: 420px;
                max-height: 420px;
                border: 1px solid #ccc;
                display: none;
            "
        >

        <p id="preview_info" style="font-family: monospace;"></p>
    </div>

</div>

<script>
    var plot = document.getElementById("{div_id}");

    plot.on("plotly_click", function(data) {{

        var point = data.points[0];

        if (!point.customdata) {{
            return;
        }}

        var imageData = point.customdata[0];
        var filename = point.customdata[1];
        var index = point.customdata[2];
        var cluster = point.customdata[3];
        var fullPath = point.customdata[4];

        if (!imageData) {{
            document.getElementById("preview_title").innerText =
                "Immagine non trovata";

            document.getElementById("preview_img").style.display = "none";

            document.getElementById("preview_info").innerText =
                "idx=" + index + "\\n" +
                "cluster=" + cluster + "\\n" +
                fullPath;

            return;
        }}

        document.getElementById("preview_title").innerText =
            "idx=" + index + " | cluster=" + cluster;

        document.getElementById("preview_img").src = imageData;
        document.getElementById("preview_img").style.display = "block";

        document.getElementById("preview_info").innerText =
            filename + "\\n" + fullPath;
    }});
</script>

</body>
</html>
"""

    output_html_path = Path(output_html_path)
    output_html_path.write_text(html, encoding="utf-8")

    print(f"File HTML creato: {output_html_path.resolve()}")

    return output_html_path

In [ ]:
output_html = create_clickable_umap_html(
    embedding=best_umap_embedding,
    labels=best_labels,
    image_paths=image_paths,
    clusters_to_show=[7],
    output_html_path="clickable_cluster_7.html",
    title=(
        f"{EMBEDDING_PATH.stem} - GMM su UMAP ottimizzato - Cluster 7"
    ),
    click_only_selected=True,
)

In [ ]:
import webbrowser

webbrowser.open(output_html.resolve().as_uri())

In [ ]:
clustered_metadata = metadata.copy()

if len(clustered_metadata) != len(best_labels):
    raise ValueError(
        f"metadata ha {len(clustered_metadata)} righe, "
        f"ma best_labels ha lunghezza {len(best_labels)}."
    )

clustered_metadata["leiden_cluster"] = best_labels

OUTPUT_DIR = EMBEDDINGS_DIR / "clustered"
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / f"{EMBEDDING_PATH.stem}_pca95_umap_leiden.csv"

clustered_metadata.to_csv(OUTPUT_PATH, index=False)

print("Salvato:", OUTPUT_PATH)
display(clustered_metadata.head())

## Sample per cluster

Usa le probabilita' posteriori del GMM per ordinare le immagini piu' rappresentative di ogni cluster.

In [ ]:
%matplotlib inline

if not image_paths:
    raise RuntimeError(
        "Colonna image_path non trovata nel CSV: "
        "impossibile mostrare i sample."
    )

sample_figure, sample_axes = show_clustering_samples(
    labels=best_labels,
    probabilities=None,
    image_paths=image_paths,
    x=15,
    include_noise=False,
    title=f"{EMBEDDING_PATH.stem} - sample per cluster Leiden",
)

SAMPLE_FIGURE_PATH = (
    PROJECT_ROOT / f"cluster_{EMBEDDING_PATH.stem}_leiden_opt.png"
)

sample_figure.savefig(
    SAMPLE_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)

print("Figura salvata:", SAMPLE_FIGURE_PATH)
#sample_figure

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image


def plot_all_samples_from_clusters(
    labels,
    image_paths,
    clusters_to_show,
    max_cols=8,
    image_size=2.2,
    title_prefix="Cluster samples",
):
    """
    Visualizza tutti i sample appartenenti ai cluster selezionati.

    Parametri
    ----------
    labels : array-like
        Label di clustering, shape = (n_samples,).

    image_paths : list-like
        Lista dei path delle immagini nello stesso ordine degli embedding.

    clusters_to_show : list
        Lista delle label dei cluster da visualizzare.

    max_cols : int
        Numero massimo di colonne nella griglia.

    image_size : float
        Dimensione di ogni immagine nella figura.

    title_prefix : str
        Titolo principale del plot.
    """

    labels = np.asarray(labels)

    if len(labels) != len(image_paths):
        raise ValueError(
            f"labels ha lunghezza {len(labels)}, "
            f"image_paths ha lunghezza {len(image_paths)}. "
            "Devono avere la stessa lunghezza."
        )

    print("Cluster presenti:", np.unique(labels))
    print("Cluster richiesti:", clusters_to_show)

    for cluster_id in clusters_to_show:
        cluster_indices = np.where(labels == cluster_id)[0]

        n_samples = len(cluster_indices)

        if n_samples == 0:
            print(f"Cluster {cluster_id}: nessun sample trovato.")
            continue

        print(f"Cluster {cluster_id}: {n_samples} sample")

        n_cols = min(max_cols, n_samples)
        n_rows = math.ceil(n_samples / n_cols)

        figure, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(image_size * n_cols, image_size * n_rows)
        )

        axes = np.array(axes).reshape(-1)

        for ax in axes:
            ax.axis("off")

        for plot_idx, sample_idx in enumerate(cluster_indices):
            image_path = Path(image_paths[sample_idx])

            try:
                image = Image.open(image_path).convert("RGB")
            except Exception as e:
                print(f"Errore apertura immagine {image_path}: {e}")
                continue

            axes[plot_idx].imshow(image)
            axes[plot_idx].axis("off")
            axes[plot_idx].set_title(
                f"idx={sample_idx}",
                fontsize=8
            )

        figure.suptitle(
            f"{title_prefix} - cluster {cluster_id} "
            f"({n_samples} sample)",
            fontsize=14
        )

        figure.tight_layout()
        plt.show()

clusters_to_show = [5]

masked_dir = Path("/home/pasquale/PycharmProjects/Glomeruli-FP03-2026/data/glomeruli/masked")

image_paths_masked = sorted(
    list(masked_dir.glob("*.png")) +
    list(masked_dir.glob("*.jpg")) +
    list(masked_dir.glob("*.jpeg")) +
    list(masked_dir.glob("*.tif")) +
    list(masked_dir.glob("*.tiff"))
)

print("Numero immagini:", len(image_paths_masked))
print("Numero labels:", len(best_labels))
plot_all_samples_from_clusters(
    labels=best_labels,
    image_paths=image_paths_masked,
    clusters_to_show=clusters_to_show,
    max_cols=8,
    image_size=2.0,
    title_prefix=f"{EMBEDDING_PATH.stem} - GMM"
)

In [ ]:
from sklearn.metrics import silhouette_samples
import pandas as pd
import numpy as np


def per_cluster_silhouette(X, labels):
    X = np.asarray(X)
    labels = np.asarray(labels)

    sil_values = silhouette_samples(X, labels)

    df = pd.DataFrame({
        "cluster": labels,
        "silhouette": sil_values,
    })

    summary = (
        df.groupby("cluster")
        .agg(
            mean_silhouette=("silhouette", "mean"),
            median_silhouette=("silhouette", "median"),
            min_silhouette=("silhouette", "min"),
            max_silhouette=("silhouette", "max"),
            n_samples=("silhouette", "size"),
        )
        .sort_values("mean_silhouette")
    )

    return summary

silhouette_summary = per_cluster_silhouette(
    X=best_umap_embedding,
    labels=leiden_optimization["best_labels"],
)

silhouette_summary